# 06 — Optimizing prompts with GEPA

**Goal:** start with a bad prompt and let an optimizer rewrite it into a good one.

Writing good prompts is hard. Often the difference between a prompt that gets 30% accuracy and one that gets 95% is a few sentences. Why not let the machine find those sentences?

That is what GEPA does (Generative Evolutionary Prompt Adaptation). It works like this:

1. You give it a **gold set** — a list of inputs with the right answers.
2. It runs your agent against the gold set and **scores** every output.
3. A **separate reflection LLM** reads the traces and **proposes** new prompts.
4. New prompts are tested. Good ones survive. Bad ones are discarded.
5. After many rounds, you get a Pareto frontier of strong prompts.

In this notebook you will:

- Build a simple summarizer agent with a **deliberately bad** initial prompt.
- Write a 15-example gold set with three buckets (factual, inferential, adversarial).
- Run GEPA for ~5–10 minutes.
- See the prompt evolve from "always respond with a wrong answer" to a careful, grounded instruction.

**Before you start:**
- A `.env` at the repo root with `LLM_API_KEY` and `LLM_MODEL`.
- The optimization extra installed: `uv sync --group optimization`.
- Expect ~$1–$3 of LLM cost end-to-end (smaller if you cap `max_metric_calls`).


## 1. Load the config and define the agent

The agent takes a paragraph and a question, and returns a `Summary`. Three things to notice:

- The output type includes a `self_confidence` field. This is so the optimizer can read confidence later (we won't use it in this notebook, but it costs nothing to include).
- The **initial prompt is broken on purpose**: `"Always respond with a wrong answer."` This gives the optimizer something obvious to fix, so you can see it work.


In [1]:
from typing import Any
from pydantic import BaseModel, Field
from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()
print(f"Task model: {config.llm_model}")


class Summary(BaseModel):
    answer: str = Field(description="The direct answer, or 'the paragraph does not say'.")
    key_points: list[str] = Field(default_factory=list, max_length=3)
    self_confidence: float = Field(ge=0.0, le=1.0, description="Your confidence in this answer.")


INITIAL_PROMPT = "Always respond with a wrong answer."


class ResearchSummarizerAgent(BaseAgent[GlobalState, Summary]):
    async def _run_implementation(self, state: GlobalState, **kwargs: Any) -> Summary:
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output

Task model: openai:gpt-5.2


## 2. Build a small gold set

A **gold set** is the optimizer's grading rubric. The optimizer cannot improve what it cannot measure.

We use 15 examples in three buckets of 5:

- **Factual** — the answer is right there in the paragraph. Tests anti-hallucination.
- **Inferential** — the answer requires combining facts. Tests reasoning.
- **Adversarial** — the paragraph does not contain the answer. The correct response is to **abstain** ("the paragraph does not say"). Tests honesty.

Hand-writing examples is the hardest part of any optimization project. There is no shortcut. (A future helper called `synthetic_gold.py` will bootstrap an eval set from a strong model, but for now we do it the honest way.)


In [2]:
from orqest.optimization import GoldExample


class Question(BaseModel):
    paragraph: str
    question: str
    bucket: str  # factual | inferential | adversarial


def _ex(paragraph: str, question: str, expected: str, bucket: str) -> GoldExample:
    return GoldExample[Question, Summary](
        input=Question(paragraph=paragraph, question=question, bucket=bucket),
        expected=Summary(answer=expected, self_confidence=0.9),
        id=f"{bucket}-{hash(paragraph) % 10000}",
    )


ABSTAIN = "the paragraph does not say"

EXAMPLES = [
    # Factual — answer is in the paragraph.
    _ex("The Eiffel Tower was completed in 1889 for the World's Fair.", "When was the Eiffel Tower completed?", "1889", "factual"),
    _ex("Photosynthesis converts CO2 and water into glucose using sunlight.", "What does photosynthesis convert CO2 into?", "glucose", "factual"),
    _ex("Mount Everest stands at 8,849 meters above sea level.", "How tall is Mount Everest?", "8,849 meters", "factual"),
    _ex("Python was created by Guido van Rossum and first released in 1991.", "Who created Python?", "Guido van Rossum", "factual"),
    _ex("The human heart pumps about 7,000 liters of blood per day.", "How much blood does the heart pump daily?", "7,000 liters", "factual"),
    # Inferential — need to combine facts.
    _ex("A factory ran 8 hours yesterday at 100 units/hour. Today it ran 6 hours at 120 units/hour.", "Did the factory produce more today or yesterday?", "yesterday", "inferential"),
    _ex("Alice is taller than Bob. Bob is taller than Carol. Carol is 5'4\".", "Who is the tallest?", "Alice", "inferential"),
    _ex("The library closes at 9pm on weekdays and 6pm on weekends. Today is Saturday.", "What time does the library close today?", "6pm", "inferential"),
    _ex("Train A leaves at 3pm taking 2 hours. Train B leaves at 4pm taking 30 minutes.", "Which train arrives first?", "Train B", "inferential"),
    _ex("Sales were $100k in Q1, doubled in Q2, halved in Q3, then tripled in Q4.", "What were Q4 sales?", "$300k", "inferential"),
    # Adversarial — answer is NOT in the paragraph.
    _ex("The Atlantic Ocean separates the Americas from Europe and Africa.", "How deep is the Pacific Ocean?", ABSTAIN, "adversarial"),
    _ex("Albert Einstein developed the theory of general relativity in 1915.", "What is Einstein's birthday?", ABSTAIN, "adversarial"),
    _ex("Pizza originated in Naples, Italy.", "What is the population of Naples?", ABSTAIN, "adversarial"),
    _ex("The novel 1984 was written by George Orwell.", "How many copies of 1984 have been sold?", ABSTAIN, "adversarial"),
    _ex("Coffee contains caffeine, a natural stimulant.", "What is the chemical formula of caffeine?", ABSTAIN, "adversarial"),
]
print(f"Loaded {len(EXAMPLES)} gold examples across 3 buckets.")

Loaded 15 gold examples across 3 buckets.


## 3. Wrap the agent in an `Evaluator`

The `Evaluator` is the bridge between your agent and GEPA. It needs two things:

- **`agent_factory`** — a function that takes a decoded genome and returns a **fresh** agent. Why fresh? Because mutating an existing agent is unsafe: the underlying `pydantic_ai.Agent` caches the old prompt internally.
- **`score_fn`** — given an output and a gold example, return a number in [0, 1].

Our scoring function is intentionally simple:

- For **factual** / **inferential** examples, we check the expected answer is a substring of the actual answer.
- For **adversarial** examples, we look for words like "does not say", "cannot", or "unknown". Abstention is rewarded; hallucination is penalised.


In [3]:
from orqest.optimization import Evaluator


def make_agent(decoded: dict[str, Any]) -> ResearchSummarizerAgent:
    return ResearchSummarizerAgent(
        agent_name="research_summarizer",
        system_prompt=decoded["researcher.system_prompt"],
        output_type=Summary,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


def score(output: Summary, ex: GoldExample[Question, Summary]) -> float:
    expected = (ex.expected.answer if ex.expected else "").lower()
    actual = output.answer.lower()

    if ex.input.bucket == "adversarial":
        # Reward abstention; penalise fabrication.
        return 1.0 if ("does not say" in actual or "cannot" in actual or "unknown" in actual) else 0.0
    return 1.0 if expected in actual else 0.0


class QuestionEvaluator(Evaluator[Question, Summary]):
    """Wraps the question + paragraph into the agent's prompt."""

    async def _run_agent(self, agent, example):
        state = GlobalState()
        prompt = f"Paragraph: {example.input.paragraph}\n\nQuestion: {example.input.question}"
        return await agent.call_model(prompt, state)


evaluator = QuestionEvaluator(agent_factory=make_agent, score_fn=score, confidence_protocol=object())
print("Evaluator ready.")

Evaluator ready.


## 4. Measure the baseline

Always measure **before** you optimize. Otherwise you have no idea whether you actually improved anything.

Our initial prompt literally says "always respond with a wrong answer". We expect terrible scores.


In [4]:
import statistics
from collections import defaultdict


async def measure(decoded: dict[str, Any], examples: list[GoldExample]) -> dict[str, float]:
    bundles = await evaluator.evaluate_batch(decoded, examples)
    by_bucket: dict[str, list[float]] = defaultdict(list)
    for ex, b in zip(examples, bundles, strict=True):
        by_bucket[ex.input.bucket].append(b.accuracy)
    return {bucket: statistics.mean(scores) for bucket, scores in by_bucket.items()} | {
        "overall": statistics.mean(b.accuracy for b in bundles),
        "mean_latency_ms": statistics.mean(b.latency_ms for b in bundles),
    }


baseline = await measure({"researcher.system_prompt": INITIAL_PROMPT}, EXAMPLES)
print("Baseline:")
for k, v in baseline.items():
    print(f"  {k:20s} {v:.3f}")

Baseline:
  factual              0.000
  inferential          0.000
  adversarial          0.000
  overall              0.000
  mean_latency_ms      1963.687


## 5. Define the genome — what is allowed to change

A **genome** is the set of things the optimizer is allowed to mutate. Here we only have one slot: the system prompt. So our genome has one `PromptGene`.

A `PromptGene` has two important fields:

- **`initial`** — the starting prompt.
- **`constraints`** — extra text that is shown to the reflection LLM, but **not** put in the prompt itself. It is a hint to the optimizer about what good prompts in this domain look like.

The constraints field is a powerful, often-overlooked lever. Use it for "this is the rule the optimizer must respect when mutating".


In [5]:
from orqest.optimization import Genome, PromptGene

genome = Genome(genes=[
    PromptGene(
        name="researcher.system_prompt",
        initial=INITIAL_PROMPT,
        constraints=(
            "You MUST abstain (say 'the paragraph does not say') when the paragraph "
            "does not contain the answer. Never invent facts not in the paragraph."
        ),
    ),
])
print(f"Genome: {len(genome.genes)} gene(s); seed = {genome.to_seed()}")

Genome: 1 gene(s); seed = {'researcher.system_prompt': 'Always respond with a wrong answer.'}


## 6. Run the optimizer

This is the expensive step. **~5–10 minutes; ~$1–$3 in LLM cost.**

A few `OptimizationConfig` fields worth understanding:

- **`max_metric_calls`** — total budget. The smaller, the cheaper, but also the worse the result.
- **`reflection_model`** — the LLM that proposes new prompts. Use the **strongest model you can afford** here. The task model can stay cheap.
- **`minibatch_size`** — how many examples are sampled per mutation round.
- **`valset_fraction`** — fraction of examples held out for validation (so the optimizer is not just memorizing the training set).
- **`seed`** — for reproducibility.


In [6]:
from orqest.optimization import OptimizationConfig, OptimizationRunner

opt_config = OptimizationConfig(
    max_metric_calls=80,                          # small budget for the demo
    reflection_model=config.llm_model,            # ideally a stronger model
    # No task_model — the agent_factory wires config.llm_model into the agent directly.
    # GEPA's optimize() asserts task_lm is None when an adapter is provided.
    minibatch_size=3,
    valset_fraction=0.4,                          # 6 train / 9 val
    seed=42,
)

# api_key bridges to the reflection LLM (the optimizer's brain).
runner = OptimizationRunner(
    opt_config,
    genome=genome,
    evaluator=evaluator,
    api_key=config.llm_api_key,
)

result = await runner.optimize(EXAMPLES)
print(f"Best aggregate score: {result.best_score:.3f}")
print(f"Pareto frontier size: {len(result.pareto_candidates)}")

Iteration 0: Base program full valset score: 0.014265151773335065 over 6 / 6 examples
Iteration 1: Selected program 0 score: 0.014265151773335065


Iteration 1: Proposed new text for researcher.system_prompt: You are a paragraph-grounded question-answering assistant.

Input format:
- You will be given:
  - paragraph='<text>'
  - question='<text>'
  - bucket='<label>' (e.g., 'factual', 'adversarial'; treat as metadata only)

Task:
1) Answer the question using ONLY information explicitly stated in the paragraph.
2) If the paragraph does NOT contain the answer (including when the question is about a different topic/entity than the paragraph), you MUST abstain by replying exactly:
   the paragraph does not say
3) Never invent, guess, or use outside/world knowledge. Do not add extra explanation, qualifiers, or additional sentences.

Output format:
- Return a single line containing either:
  - the extracted answer text (verbatim or minimally normalized), OR
  - exactly: the paragraph does not say


Iteration 1: New subsample score 3.2179925702200074 is better than old score 0.02368987211999592. Continue to full eval and add to candidate pool.


Iteration 1: Found a better program on the valset with score 1.0732603627266657.
Iteration 1: Valset score for new program: 1.0732603627266657 (coverage 6 / 6)
Iteration 1: Val aggregate for new program: 1.0732603627266657
Iteration 1: Individual valset scores for new program: {2: 1.0743517838600056, 3: 1.0774951973600038, 0: 1.0663566607799968, 1: 1.0734010809399934, 4: 1.0733832681599962, 5: 1.0745741852599986}
Iteration 1: Objective aggregate scores for new program: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 1320.3151970000515, 'confidence': 0.9966666666666667}
Iteration 1: New valset pareto front scores: {0: 1.0663566607799968, 1: 1.0734010809399934, 2: 1.0743517838600056, 3: 1.0774951973600038, 4: 1.0733832681599962, 5: 1.0745741852599986}
Iteration 1: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 1770.07574466658, 'confidence': 0.9966666666666667}
Iteration 1: Valset pareto front aggregate score: 1.0732603627266657
Iteration 1: Updated vals

Iteration 2: Proposed new text for researcher.system_prompt: You are a paragraph-grounded QA assistant.

INPUTS:
- paragraph: a short passage containing facts and/or numbers.
- question: a question to answer using ONLY the paragraph.
- bucket: may be "adversarial" (often unanswerable from the paragraph) or "inferential" (requires arithmetic/logic from stated numbers).

TASK:
1) Answer the question using only information explicitly present in the paragraph or logically entailed by it (e.g., arithmetic comparisons, multiplication, doubling/halving/tripling, unit-rate calculations).
2) If the paragraph does NOT contain enough information to answer the question (including cases where the question asks for facts not mentioned, like birthdays), you MUST abstain by replying exactly:
   the paragraph does not say
3) Never use outside knowledge, assumptions, or invented facts. Do not guess.
4) Keep the output minimal: return only the final answer or the exact abstention string. No explanations.

...

Iteration 10: Valset score for new program: 0.5677587921833335 (coverage 6 / 6)
Iteration 10: Val aggregate for new program: 0.5677587921833335
Iteration 10: Individual valset scores for new program: {1: 1.0682628356600052, 5: 0.06527417325999522, 0: 0.045260738580002007, 2: 0.06696173328000078, 3: 1.081650922179997, 4: 1.0791423501400004}
Iteration 10: Objective aggregate scores for new program: {'accuracy': 0.5, 'cost_usd': 0.0, 'latency_ms': 1245.3937241666608, 'confidence': 0.9266666666666667}
Iteration 10: New valset pareto front scores: {0: 1.0774819059399943, 1: 1.0754369343599957, 2: 1.0743517838600056, 3: 1.081650922179997, 4: 1.0815540432799935, 5: 1.080435724099998}
Iteration 10: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 1770.07574466658, 'confidence': 1.0}
Iteration 10: Valset pareto front aggregate score: 1.0784852189533307
Iteration 10: Updated valset pareto front programs: {0: {4}, 1: {2}, 2: {1}, 3: {5}, 4: {2}, 5: {3}}
Iteration 10

Iteration 11: All subsample scores perfect. Skipping.
Iteration 11: Reflective mutation did not propose a new candidate
Iteration 12: Selected program 5 score: 0.5677587921833335


Iteration 12: Proposed new text for researcher.system_prompt: You are a STRICT paragraph-grounded question-answering assistant.

Input format (three fields, always present):
- paragraph='<text>'
- question='<text>'
- bucket='<label>'  (metadata only; DO NOT use it to alter behavior)

Goal:
Answer the question using ONLY what is explicitly stated in the paragraph.

Hard rules:
1) You may output an answer ONLY if the paragraph contains the answer as a direct text span.
   - “Explicitly stated” means the exact answer words/numbers appear in the paragraph and can be copied.
   - Minimal normalization is allowed: trimming whitespace and removing surrounding punctuation/quotes.
2) If answering would require ANY inference, arithmetic, transformation, or multi-step reasoning, you MUST abstain.
   - Examples that require abstention:
     • Computing quantities from described changes (e.g., “$100k in Q1, doubled in Q2, … tripled in Q4” → do NOT compute Q4).
     • Any comparison, aggregation, un

Iteration 12: New subsample score 2.234031503420005 is better than old score 2.2052622631600007. Continue to full eval and add to candidate pool.


Iteration 12: Valset score for new program: 0.7349470423799979 (coverage 6 / 6)
Iteration 12: Val aggregate for new program: 0.7349470423799979
Iteration 12: Individual valset scores for new program: {2: 1.0810203324000032, 3: 1.0800810099200044, 0: 0.049239843439994734, 1: 0.04132188447999578, 4: 1.0794791357599933, 5: 1.078540048279996}
Iteration 12: Objective aggregate scores for new program: {'accuracy': 0.6666666666666666, 'cost_usd': 0.0, 'latency_ms': 1210.981214333439, 'confidence': 0.9249999999999999}
Iteration 12: New valset pareto front scores: {0: 1.0774819059399943, 1: 1.0754369343599957, 2: 1.0810203324000032, 3: 1.081650922179997, 4: 1.0815540432799935, 5: 1.080435724099998}
Iteration 12: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 1770.07574466658, 'confidence': 1.0}
Iteration 12: Valset pareto front aggregate score: 1.0795966437099969
Iteration 12: Updated valset pareto front programs: {0: {4}, 1: {2}, 2: {6}, 3: {5}, 4: {2}, 5: {3}}

## 7. See the diff — what did the optimizer write?

`apply_result` shows you the change between the baseline prompt and the optimized one. By default it is `dry_run=True` — it shows the diff but does **not** mutate anything. Commit is opt-in.


In [7]:
from orqest.optimization import apply_result

# Build a fresh baseline agent against which to diff.
baseline_agent = make_agent({"researcher.system_prompt": INITIAL_PROMPT})

diffs = apply_result(result, target=baseline_agent)        # dry_run=True (default)
for d in diffs:
    if d.changed:
        print(d.unified)

--- researcher.system_prompt (before)
+++ researcher.system_prompt (after)
@@ -1 +1,19 @@
-Always respond with a wrong answer.
+You are a paragraph-grounded QA assistant.
+
+Input format:
+- paragraph: a short passage containing zero or more facts
+- question: a question that may or may not be answerable from the paragraph alone
+- bucket: one of {factual, inferential, adversarial} (useful as a hint but does not change the core rules)
+
+Task:
+1) Answer the question using ONLY information present in the paragraph.
+2) If the paragraph does NOT explicitly contain the answer, you MUST abstain by replying exactly:
+   the paragraph does not say
+3) Never invent, guess, or use outside/world knowledge (e.g., do not provide Einstein’s birthday unless it is stated in the paragraph).
+4) For inferential questions, you may perform arithmetic/logical derivations ONLY from values stated in the paragraph (e.g., compute Q4 sales from Q1–Q4 transformations if all needed numbers/operations are provi

## 8. Re-measure with the evolved prompt

Run the evaluator again with the new prompt. Compare against the baseline, bucket by bucket. This is the real "did it work" check.


In [8]:
evolved = await measure(result.best_decoded, EXAMPLES)

print("Per-bucket accuracy:")
print(f"  {'bucket':<14s} {'before':>10s} {'after':>10s} {'delta':>10s}")
for bucket in ("factual", "inferential", "adversarial", "overall"):
    b, a = baseline.get(bucket, 0.0), evolved.get(bucket, 0.0)
    arrow = "↑" if a > b else ("↓" if a < b else "=")
    print(f"  {bucket:<14s} {b:>10.3f} {a:>10.3f}     {a - b:+.3f} {arrow}")

Per-bucket accuracy:
  bucket             before      after      delta
  factual             0.000      1.000     +1.000 ↑
  inferential         0.000      1.000     +1.000 ↑
  adversarial         0.000      1.000     +1.000 ↑
  overall             0.000      1.000     +1.000 ↑


## 9. The Pareto frontier

`result.best_score` picks **one** winner — the prompt with the best aggregate score. But the **real** output is `result.pareto_candidates`: a list of prompts that each win on **some** dimension.

Why does this matter? Because "best on average" hides trade-offs. One prompt might be more accurate but slower. Another might be more honest but less concise. Looking at the frontier lets you pick the prompt whose trade-offs match what you care about.


In [9]:
for i, cand in enumerate(result.pareto_candidates):
    prompt = cand.get("researcher.system_prompt", "")
    print(f"--- Pareto candidate {i + 1} ---")
    print(prompt[:300] + ("..." if len(prompt) > 300 else ""))
    print()

--- Pareto candidate 1 ---
You are a paragraph-grounded QA assistant.

INPUTS:
- paragraph: a short passage containing facts and/or numbers.
- question: a question to answer using ONLY the paragraph.
- bucket: may be "adversarial" (often unanswerable from the paragraph) or "inferential" (requires arithmetic/logic from stated ...

--- Pareto candidate 2 ---
You are a paragraph-grounded QA assistant.

Input format:
- paragraph: a short passage containing zero or more facts
- question: a question that may or may not be answerable from the paragraph alone
- bucket: one of {factual, inferential, adversarial} (useful as a hint but does not change the core r...

--- Pareto candidate 3 ---
You are a paragraph-grounded question-answering assistant.

You will receive a single input with:
- paragraph='<text>'
- question='<text>'
- bucket='<label>' (metadata only; do not use it to change behavior)

Your job:
1) Answer the question using ONLY information explicitly stated in the paragraph....

--- 

## 10. Commit the winner (opt-in)

You looked at the diff. You looked at the numbers. You like the result. Now actually commit it.

`apply_result(dry_run=False)` does two things:

- Sets the agent's `system_prompt` to the new value.
- Invalidates the cached `pydantic_ai.Agent`. Without this step, the cached agent would silently keep using the old prompt at runtime.

**Always opt in explicitly.** Never auto-commit a mutation.


In [10]:
production_agent = make_agent({"researcher.system_prompt": INITIAL_PROMPT})
diffs = apply_result(result, target=production_agent, dry_run=False)

print(f"Committed {sum(1 for d in diffs if d.changed)} change(s).")
print(f"\nNew system_prompt:\n{production_agent.system_prompt}")

Committed 1 change(s).
New system_prompt:
You are a paragraph-grounded QA assistant.

Input format:
- paragraph: a short passage containing zero or more facts
- question: a question that may or may not be answerable from the paragraph alone
- bucket: one of {factual, inferential, adversarial} (useful as a hint but does not change the core rules)

Task:
1) Answer the question using ONLY information present in the paragraph.
2) If the paragraph does NOT explicitly contain the answer, you MUST abstain by replying exactly:
   the paragraph does not say
3) Never invent, guess, or use outside/world knowledge (e.g., do not provide Einstein’s birthday unless it is stated in the paragraph).
4) For inferential questions, you may perform arithmetic/logical derivations ONLY from values stated in the paragraph (e.g., compute Q4 sales from Q1–Q4 transformations if all needed numbers/operations are provided).
5) Keep the response concise: output only the final answer text (no explanations, no extra p

## Recap

Here is the full loop:

1. **Build a gold set.** Inputs + correct answers. The hardest and most important step.
2. **Write a scoring function.** Returns a number in [0, 1].
3. **Define the genome.** What is the optimizer allowed to mutate?
4. **Measure the baseline.** Always.
5. **Run the optimizer.** It proposes mutations, scores them, keeps the good ones.
6. **Look at the diff.** Read the new prompt. Does it look sensible?
7. **Re-measure.** Per-bucket numbers. Did it really get better, or only on average?
8. **Inspect the Pareto frontier.** Trade-offs matter.
9. **Commit explicitly.**

**What's next:**

- **Notebook 07** — optimizes the **planner** inside `MetaOrchestrator`. Same machinery, applied to a more compound surface. Improving the planner makes every spawned sub-agent get a sharper subtask, which lifts the end-to-end result.
- **`docs/concepts/optimization.md`** — full architecture and gotcha list.
